In [0]:
# =======================================================================
# Notebook 03: EDA & Feature Engineering
# =======================================================================

print("=" * 80)
print("Notbook 03: Feature Engineering")
print("="* 80)

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import builtins
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print()

In [0]:
# ===========================================================
#  Load Cleaned Data from Notebook 02
# ===========================================================

print(" Loading Cleaned Delta Table")
print("="* 80)

delta_path = "/Volumes/workspace/lending_club/lending_club_data/cleaned_loans_delta/"

df = spark.read.format("delta").load(delta_path)

print(f"Delta table loaded successfully")
print(f"Rows : {df.count()}")
print(f"Columns: {len(df.columns)}")
print()


In [0]:
display(df)

In [0]:
# Column renaming
rename_dict = {
    # time-based
    'mths_since_recent_inq': 'months_since_last_inquiry',
    'mths_since_recent_bc': 'months_since_last_credit_card_activity',
    'mo_sin_old_il_acct': 'months_since_oldest_installment_account',
    'mo_sin_old_rev_tl_op': 'months_since_oldest_revolving_account',
    'mo_sin_rcnt_rev_tl_op': 'months_since_recent_revolving_account',
    'mo_sin_rcnt_tl': 'months_since_recent_account',

    # credit card features
    'bc_util': 'credit_card_utilisation_pct',
    'percent_bc_gt_75': 'pct_credit_cards_over_75_util',
    'bc_open_to_buy': 'credit_card_available_limit',

    # inquiries
    'inq_fi': 'financial_inquiries'
}

for old,new in rename_dict.items():
    if old in df.columns:
        df = df.withColumnRenamed(old,new)

print("Renamed selected columns")

In [0]:
# ======================================================================
# Rename Columns for Clarity
# ======================================================================

print("Renaming columns for clarity...")
print("=" * 80)

df = df \
    .withColumnRenamed('delinquencies_2yrs', 'delinquencies_last_2yrs') \
    .withColumnRenamed('fico_range_low', 'fico_score_low') \
    .withColumnRenamed('fico_range_high', 'fico_score_high') \
    .withColumnRenamed('last_fico_range_low', 'last_fico_score_low') \
    .withColumnRenamed('last_fico_range_high', 'last_fico_score_high') \
    .withColumnRenamed('inquiries_last_6mths', 'credit_inquiries_6mths') \
    .withColumnRenamed('open_accounts', 'open_credit_accounts') \
    .withColumnRenamed('public_records', 'derogatory_public_records') \
    .withColumnRenamed('revolving_balance', 'total_revolving_balance') \
    .withColumnRenamed('total_accounts', 'total_credit_accounts') \
    .withColumnRenamed('outstanding_principal', 'remaining_principal') \
    .withColumnRenamed('outstanding_principal_inv', 'remaining_principal_inv') \
    .withColumnRenamed('collections_12_mths_ex_med', 'collections_last_12mths') \
    .withColumnRenamed('accounts_now_delinquent', 'currently_delinquent_accounts') \
    .withColumnRenamed('acc_open_past_24mths', 'accounts_opened_24mths') \
    .withColumnRenamed('chargeoff_within_12_mths', 'chargeoffs_last_12mths') \
    .withColumnRenamed('delinq_amnt', 'delinquent_amount') \
    .withColumnRenamed('num_accts_ever_120_pd', 'accounts_ever_120dpd') \
    .withColumnRenamed('num_actv_bc_tl', 'active_bankcard_accounts') \
    .withColumnRenamed('num_actv_rev_tl', 'active_revolving_accounts') \
    .withColumnRenamed('num_bc_sats', 'satisfactory_bankcard_accounts') \
    .withColumnRenamed('num_bc_tl', 'total_bankcard_accounts') \
    .withColumnRenamed('num_il_tl', 'total_installment_accounts') \
    .withColumnRenamed('num_op_rev_tl', 'open_revolving_accounts') \
    .withColumnRenamed('num_rev_accts', 'total_revolving_accounts') \
    .withColumnRenamed('num_rev_tl_bal_gt_0', 'revolving_accounts_with_balance') \
    .withColumnRenamed('num_sats', 'satisfactory_accounts') \
    .withColumnRenamed('num_tl_120dpd_2m', 'accounts_120dpd_last_2mths') \
    .withColumnRenamed('num_tl_30dpd', 'accounts_30dpd') \
    .withColumnRenamed('num_tl_90g_dpd_24m', 'accounts_90dpd_last_24mths') \
    .withColumnRenamed('num_tl_op_past_12m', 'accounts_opened_last_12mths') \
    .withColumnRenamed('pct_tl_nvr_dlq', 'pct_accounts_never_delinquent') \
    .withColumnRenamed('bankruptcy_records', 'public_bankruptcies') \
    .withColumnRenamed('total_bal_ex_mort', 'total_balance_excl_mortgage') \
    .withColumnRenamed('total_il_high_credit_limit', 'total_installment_credit_limit') \
    .withColumnRenamed('financial_inquiries', 'finance_company_inquiries') \
    .withColumnRenamed('open_acc_6m', 'accounts_opened_6mths') \
    .withColumnRenamed('open_act_il', 'active_installment_accounts') \
    .withColumnRenamed('open_il_12m', 'installment_accounts_opened_12mths') \
    .withColumnRenamed('open_il_24m', 'installment_accounts_opened_24mths') \
    .withColumnRenamed('open_rv_12m', 'revolving_accounts_opened_12mths') \
    .withColumnRenamed('open_rv_24m', 'revolving_accounts_opened_24mths') \
    .withColumnRenamed('max_bal_bc', 'max_bankcard_balance') \
    .withColumnRenamed('total_cu_tl', 'total_finance_trades') \
    .withColumnRenamed('inq_last_12m', 'credit_inquiries_12mths') \
    .withColumnRenamed('avg_cur_bal', 'avg_current_balance') \
    .withColumnRenamed('total_rev_hi_lim', 'total_revolving_credit_limit') \
    .withColumnRenamed('all_util', 'overall_utilisation') \
    .withColumnRenamed('il_util', 'installment_utilisation') \
    .withColumnRenamed('total_bal_il', 'total_installment_balance') \
    .withColumnRenamed('max_bal_bc', 'max_bankcard_balance') \
    .withColumnRenamed('mortgage_accounts', 'total_mortgage_accounts') \
    .withColumnRenamed('tax_liens', 'tax_lien_records')

print(f"✓ Columns renamed")
print(f"Total columns: {len(df.columns)}")
print()
print("Updated column list:")
print(df.columns)

In [0]:
# ====================================================
# : EDA - Overview Statistics
# ====================================================

print(" EDA - Overview statistics")
print("="* 80)

# Get all numeric columns
numeric_cols = [c for c,t in df.dtypes if t in ('double','int','bigint')                                                                                                          
]

print("Summary statistics for key numeric columns")
df.select(numeric_cols).describe().display(truncate=False)

### EDA Observations: Outlier & Data Quality Issues Found

After examining all **84 numeric columns**, the following issues were identified
and resolved before feature engineering.

---

### 🔴 Impossible Values — Set to NULL → Imputed Median

These values are statistically impossible and represent data entry errors.
Setting to NULL and imputing median preserves row count without introducing
fake data.

| Column | Issue | Action |
|--------|-------|--------|
| `debt_to_income` | min: -1.0, max: 999.0 — DTI can never be negative or 999% | NULL → median |
| `months_since_oldest_installment_account` | max: 999.0 — cannot be 999 months | NULL → median |
| `months_since_oldest_revolving_account` | max: 999.0 — same issue as above | NULL → median |
| `installment_utilisation` | max: 1000.0 — utilisation cannot exceed 1000% | NULL → median |
| `total_rec_late_fee` | min: -9.5E-9 — fees cannot be negative | → 0 |

---

### 🟡 Extreme But Valid Outliers — Kept As-Is

These values are extreme but financially possible. Bucketing in feature
engineering neutralises their ability to skew analysis.

| Column | Issue | Action |
|--------|-------|--------|
| `annual_income` | max: £110,000,000 | Keep → bucketed into Q4 `income_band` |
| `total_revolving_credit_limit` | max: 9,999,999 | Keep → not used in continuous visuals |
| `total_high_credit_limit` | max: 9,999,999 | Keep → not used in continuous visuals |
| `total_revolving_balance` | max: 2,904,836 | Keep → bucketed |
| `total_collection_amount` | max: 6,214,661 | Keep → bucketed |

---

### 🟠 Capped at Boundary — Power BI Visual Distortion

These columns are used as continuous axes in Power BI charts. Without
capping, a handful of extreme values stretch the axis and make the
chart unreadable.

| Column | Issue | Action |
|--------|-------|--------|
| `revolving_utilisation` | max: 193.0 — utilisation > 100% distorts visuals | Capped at 100 |
| `overall_utilisation` | max: 239.0 | Capped at 100 |
| `credit_card_utilisation_pct` | max: 318.2 | Capped at 100 |
| `open_credit_accounts` | max: 101.0 | Capped at 99th percentile |
| `derogatory_public_records` | max: 86.0 | Capped at 99th percentile |
| `tax_lien_records` | max: 85.0 | Capped at 99th percentile |

---

### 🗑️ Dropped — Zero Analytical Value

| Column | Issue | Action |
|--------|-------|--------|
| `policy_code` | Every single value is 1.0 — zero variation | Dropped |

---

### ✅ Clean Columns
All remaining numeric columns are within expected ranges and require
no further treatment before feature engineering.

### `debt_to_income` — Impossible Values Fixed

**Issue:** min: -1.0, max: 999.0  
**Why impossible:** DTI cannot be negative or exceed 100% for a standard loan applicant  
**Action:** Values < 0 and > 100 set to NULL → replaced with median

In [0]:
# ===============================================================================
# Outlier Fixing - Based on EDA Findings
# ===============================================================================

print("Outlier Fixing")
print("="*80)

# -------------------------------------------------------------------
# 1. Impossible values → NULL → impute median
# -------------------------------------------------------------------

print("Step 1: Setting imposssible values to NULL and imputing median")
print("-"*60)

#debt_to_income
invalid_dti = df.filter(
    (col('debt_to_income')<0) | (col('debt_to_income')>100)
).count()

df_clean = df.withColumn('debt_to_income',
                         when((col('debt_to_income')<0) |
                              (col('debt_to_income')>100),None)
                         .otherwise(col('debt_to_income'))
                         )
dti_median = df_clean.approxQuantile('debt_to_income',[0.5],0.01
                                     )[0]
df_clean = df_clean.fillna({'debt_to_income':dti_median})
print(f" debt_to_income                          — {invalid_dti:,} invalid → median: {builtins.round(dti_median, 2)}")


### EDA: months_since_oldest_installment_account

**What this column means:**
The number of months since the borrower opened their **very first 
instalment loan** — car loan, student loan, personal loan etc.
A higher value means the borrower has a longer credit history.



In [0]:
# Check distribution of months_since_oldest_installment_account
print("months_since_oldest_installment_account distribution:")
print("-" * 60)

# Stats
df_clean.select(
    min('months_since_oldest_installment_account').alias('min'),
    percentile_approx('months_since_oldest_installment_account', 0.25).alias('Q1'),
    percentile_approx('months_since_oldest_installment_account', 0.50).alias('median'),
    percentile_approx('months_since_oldest_installment_account', 0.75).alias('Q3'),
    percentile_approx('months_since_oldest_installment_account', 0.95).alias('95th_pct'),
    percentile_approx('months_since_oldest_installment_account', 0.99).alias('99th_pct'),
    max('months_since_oldest_installment_account').alias('max')
).show()

# Check high values
total_above_500 = df_clean.filter(
    col('months_since_oldest_installment_account') > 500
).count()

print(f"Total rows above 500 months: {total_above_500:,}")
print()

# Check high values
print("Value counts above 500 months (41+ years):")
df_clean.filter(col('months_since_oldest_installment_account') > 500) \
        .groupBy('months_since_oldest_installment_account') \
        .count() \
        .orderBy('months_since_oldest_installment_account', ascending=False) \
        .show()

---

### Distribution Analysis

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Min | 0 | Just opened first instalment loan |
| Q1 | 97 months | 8 years credit history |
| Median | 130 months | 11 years credit history |
| Q3 | 153 months | 13 years credit history |
| 95th percentile | 211 months | 18 years – still realistic |
| 99th percentile | 275 months | 23 years – still realistic |
| Max | 848 months | **70 years – impossible** |

---

### Issue Found

Values above **500 months (41+ years)** are suspicious:
- All appear exactly **once** — classic sign of data entry errors
- A borrower in 2018 with 500+ months credit history would have 
  opened their first loan **before 1977**
- These are isolated outliers — not a systematic pattern

**The 99th percentile (275 months = 23 years)** is a realistic 
maximum for a borrower applying for a loan in 2015–2018.

---

### Decision — Cap at 99th Percentile

**Why not set to NULL and impute median?**
These values are not impossible in the same way DTI of -1 is impossible.
They are extreme but could theoretically be real — just highly unlikely.
Capping at the 99th percentile is more appropriate because:
- It preserves the relative ordering of values
- It keeps the row in the dataset
- 275 months is a defensible business threshold
- Avoids introducing artificial median values

**Why not keep as-is?**
A single value of 848 months would stretch your Power BI axis and 
distort any visualisation using this column as a continuous variable.

**Action taken:** Capped at 99th percentile (275 months)

In [0]:
# Check current state of the column
print("Current distribution:")
df_clean.select(
    min('months_since_oldest_installment_account').alias('min'),
    percentile_approx('months_since_oldest_installment_account', 0.99).alias('p99'),
    max('months_since_oldest_installment_account').alias('max')
).show()

# Check if extreme values still exist
print("Values above 275:")
print(df_clean.filter(
    col('months_since_oldest_installment_account') > 275.0
).count())

# Check what df_clean is based on
print("Is df_clean based on df or df_renamed?")
print(df_clean.columns[:5])

In [0]:
# Get p99 threshold using percentile_approx
p99_mo_il = df_clean.select(
    percentile_approx('months_since_oldest_installment_account', 0.99)
    .alias('p99')
).collect()[0]['p99']

print(f"  99th percentile threshold: {p99_mo_il}")

# Count BEFORE fixing
invalid_mo_il = df_clean.filter(
    col('months_since_oldest_installment_account') > p99_mo_il
).count()
print(f"  Values above p99 before fix: {invalid_mo_il:,}")

# Apply fix
df_clean = df_clean.withColumn(
    'months_since_oldest_installment_account',
    when(col('months_since_oldest_installment_account') > p99_mo_il,
         p99_mo_il)
    .otherwise(col('months_since_oldest_installment_account'))
)

# Verify AFTER fixing using percentile_approx
df_clean.select(
    min('months_since_oldest_installment_account').alias('min'),
    percentile_approx('months_since_oldest_installment_account', 0.99).alias('p99'),
    max('months_since_oldest_installment_account').alias('max')
).show()

print(f"✓ months_since_oldest_installment_account — {invalid_mo_il:,} values capped at {p99_mo_il}")

### `months_since_oldest_revolving_account` — Outlier Fix

**What this column means:**
Number of months since the borrower opened their first ever 
revolving credit account — credit card, overdraft, line of credit.
A higher value means a longer revolving credit history.

In [0]:
# Check distribution first
df_clean.select(
    min('months_since_oldest_revolving_account').alias('min'),
    percentile_approx('months_since_oldest_revolving_account', 0.25).alias('Q1'),
    percentile_approx('months_since_oldest_revolving_account', 0.50).alias('median'),
    percentile_approx('months_since_oldest_revolving_account', 0.75).alias('Q3'),
    percentile_approx('months_since_oldest_revolving_account', 0.99).alias('p99'),
    max('months_since_oldest_revolving_account').alias('max')
).show()

# Check high values
print("Values above 500:")
print(df_clean.filter(
    col('months_since_oldest_revolving_account') > 500
).count())

**Distribution:**

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Min | 1 month | Just opened first revolving account |
| Median | 163 months | 13 years credit history — typical |
| Q3 | 233 months | 19 years — still realistic |
| p99 | 482 months | 40 years — borderline |
| Max | 999 months | 83 years — impossible |

**Issue found:**
- Max of 999 months = 83 years — impossible sentinel value
- 13,227 values above 500 months (41+ years) — suspicious
- All isolated single occurrences — classic data entry errors

**Action taken:** Capped at 99th percentile (482 months)

In [0]:
# Get p99 threshold
p99_mo_rev = df_clean.select(
    percentile_approx('months_since_oldest_revolving_account', 0.99)
    .alias('p99')
).collect()[0]['p99']

print(f"  99th percentile threshold: {p99_mo_rev}")

# Count before fix
invalid_mo_rev = df_clean.filter(
    col('months_since_oldest_revolving_account') > p99_mo_rev
).count()
print(f"  Values above p99 before fix: {invalid_mo_rev:,}")

# Apply fix
df_clean = df_clean.withColumn(
    'months_since_oldest_revolving_account',
    when(col('months_since_oldest_revolving_account') > p99_mo_rev,
         p99_mo_rev)
    .otherwise(col('months_since_oldest_revolving_account'))
)

# Verify after fix
df_clean.select(
    min('months_since_oldest_revolving_account').alias('min'),
    percentile_approx('months_since_oldest_revolving_account', 0.99).alias('p99'),
    max('months_since_oldest_revolving_account').alias('max')
).show()

print(f"✓ months_since_oldest_revolving_account — {invalid_mo_rev:,} values capped at {p99_mo_rev}")

### EDA: `installment_utilisation` — Distribution, Outlier Detection & Fix

### What is `installment_utilisation`?

**Definition:**
The percentage of total instalment loan credit currently still outstanding.
It measures how much of the borrower's instalment debt (car loans, student 
loans, personal loans) remains unpaid relative to the original loan amounts.



In [0]:
# Check distribution first
df_clean.select(
    min('installment_utilisation').alias('min'),
    percentile_approx('installment_utilisation', 0.25).alias('Q1'),
    percentile_approx('installment_utilisation', 0.50).alias('median'),
    percentile_approx('installment_utilisation', 0.75).alias('Q3'),
    percentile_approx('installment_utilisation', 0.99).alias('p99'),
    max('installment_utilisation').alias('max')
).show()

# Check high values
print("Values above 100:")
print(df_clean.filter(
    col('installment_utilisation') > 100
).count())

**What values mean:**

| Value | Meaning | Risk Implication |
|-------|---------|-----------------|
| 0% | All instalment loans fully paid | Very low risk |
| 1–50% | More than half paid off | Low risk |
| 51–80% | Majority still outstanding | Medium risk |
| 81–100% | Most of debt still remaining | High risk |
| >100% | Over limit due to fees or interest | Very high risk |

**Why it matters for credit risk:**
High `installment_utilisation` means the borrower still has significant 
instalment debt outstanding — leaving less financial flexibility to repay 
their Lending Club loan on time. It is a strong predictor of default risk.


In [0]:
# ======================================================================
# EDA: installment_utilisation Distribution
# ======================================================================


# Sample for visualisation
sample_pd = df_clean.select('installment_utilisation') \
                    .dropna() \
                    .sample(fraction=0.05, seed=42) \
                    .toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('installment_utilisation Distribution', 
             fontsize=14, fontweight='bold')

# Plot 1 — Full range
axes[0].hist(sample_pd['installment_utilisation'], 
             bins=100, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].set_title('Full Range (including outliers)')
axes[0].set_xlabel('Installment Utilisation')
axes[0].set_ylabel('Count')

# Plot 2 — Normal range 0-100
normal = sample_pd[sample_pd['installment_utilisation'].between(0, 100)]
axes[1].hist(normal['installment_utilisation'],
             bins=100, color='green', edgecolor='white', alpha=0.7)
axes[1].set_title('Normal Range (0-100%)')
axes[1].set_xlabel('Installment Utilisation')
axes[1].set_ylabel('Count')

# Plot 3 — Box plot
axes[2].boxplot(sample_pd['installment_utilisation'].dropna(),
                vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[2].set_title('Box Plot — shows outliers clearly')
axes[2].set_xlabel('Installment Utilisation')

plt.tight_layout()
plt.show()

# Print counts
print(f"Total rows sampled      : {len(sample_pd):,}")
print(f"Rows with value 0-100   : {len(normal):,}")
print(f"Rows with value > 100   : {len(sample_pd[sample_pd['installment_utilisation'] > 100]):,}")
print(f"Rows with value >= 1000 : {len(sample_pd[sample_pd['installment_utilisation'] >= 1000]):,}")

### `installment_utilisation` — Distribution Analysis & Fix

---

### Distribution Findings

**Plot 1 — Full Range:**
- Majority of data concentrated between 0–100%
- Small number of values extending to 250%
- Outliers visible but rare — confirms they exist without dominating

**Plot 2 — Normal Range (0–100%):**
- Strong right skew — large spike at 0%
- Many borrowers have fully paid off instalment loans (0% utilisation)
- Gradually decreasing frequency towards 100%
- This is the expected shape for instalment loan utilisation

**Plot 3 — Box Plot:**
- Box sits between Q1 (⁓30%) and Q3 (⁓80%)
- Median around 50%
- Whisker extends to ~100%
- Dots beyond whisker = outliers above 100% — few but present

---

### Summary of Findings

| Finding | Evidence |
|---------|---------|
| 96.6% of values fall within 0–100% | Plots 1 and 2 |
| 3.4% exceed 100% — real over-limit accounts | Plot 1 small bars, Plot 3 dots |
| No extreme sentinel values like 1000 found | Plot 1 axis only reaches 250 |
| Distribution is right skewed | Plot 2 — spike at 0, decreasing towards 100 |

---

### Decision — Cap at 100%

Values above 100% are **real over-limit accounts**, not data entry errors.
However they distort Power BI visual axes — a chart axis stretching to 
250% makes the normal 0–100% range unreadable.

**Action taken:** Capped at 100% — same approach as `revolving_utilisation`
and `overall_utilisation`. Preserves the over-limit meaning without 
distorting downstream visualisations.

In [0]:
# ======================================================================
# Fix: installment_utilisation — Cap at 100%
# ======================================================================

print("Fix: installment_utilisation")
print("-" * 60)

# Count before fix
high_il = df_clean.filter(col('installment_utilisation') > 100).count()
print(f"  Values above 100% before fix: {high_il:,}")

# Apply fix — cap at 100%
df_clean = df_clean.withColumn(
    'installment_utilisation',
    when(col('installment_utilisation') > 100, 100)
    .otherwise(col('installment_utilisation'))
)

# Verify using percentile_approx
df_clean.select(
    min('installment_utilisation').alias('min'),
    percentile_approx('installment_utilisation', 0.50).alias('median'),
    percentile_approx('installment_utilisation', 0.99).alias('p99'),
    max('installment_utilisation').alias('max')
).show()

print(f"✓ installment_utilisation — {high_il:,} values > 100% capped at 100")

### `total_rec_late_fee` — Distribution Analysis & Fix

**What this column means:**
The total amount of late fees collected from the borrower over the 
entire life of the loan. A late fee is charged when a borrower misses 
a payment deadline.



In [0]:
# Check distribution of total_rec_late_fee
print("total_rec_late_fee distribution:")
df_clean.select(
    min('total_rec_late_fee').alias('min'),
    percentile_approx('total_rec_late_fee', 0.50).alias('median'),
    percentile_approx('total_rec_late_fee', 0.95).alias('p95'),
    percentile_approx('total_rec_late_fee', 0.99).alias('p99'),
    max('total_rec_late_fee').alias('max')
).show()

# How many negative values?
print("Negative values:")
print(df_clean.filter(col('total_rec_late_fee') < 0).count())

# How many zero values?
print("Zero values:")
print(df_clean.filter(col('total_rec_late_fee') == 0).count())

# How many positive values?
print("Positive values (borrower was charged a late fee):")
print(df_clean.filter(col('total_rec_late_fee') > 0).count())

**Distribution findings:**

| Value | Count | % | Meaning |
|-------|-------|---|---------|
| Zero | 1,722,522 | 96.1% | Never paid a late fee — always on time |
| Positive | 69,712 | 3.9% | Paid at least one late fee |
| Negative | 6 | 0.0% | Impossible — data entry error |

**Key insight:**
96.1% of borrowers never paid a late fee — confirming the majority 
of the portfolio is performing well. Only 3.9% of borrowers have 
ever been charged a late fee.

**Issue found:**
6 rows have negative late fee values — financially impossible. 
A late fee can never be negative.

**Decision — set negative values to 0:**
- Negative values are clearly data entry errors
- Setting to 0 means "no late fee charged" — the most logical correction
- Only 6 rows affected out of 1,792,240 — negligible impact

In [0]:
# ======================================================================
# Fix: total_rec_late_fee — Set negative values to 0
# ======================================================================

print("Fix: total_rec_late_fee")
print("-" * 60)

# Count before fix
invalid_fee = df_clean.filter(col('total_rec_late_fee') < 0).count()
print(f"  Negative values before fix: {invalid_fee:,}")

# Apply fix
df_clean = df_clean.withColumn(
    'total_rec_late_fee',
    when(col('total_rec_late_fee') < 0, 0)
    .otherwise(col('total_rec_late_fee'))
)

# Verify
df_clean.select(
    min('total_rec_late_fee').alias('min'),
    percentile_approx('total_rec_late_fee', 0.99).alias('p99'),
    max('total_rec_late_fee').alias('max')
).show()

print(f"✓ total_rec_late_fee — {invalid_fee:,} negative values set to 0")

### Utilisation Columns — Distribution Analysis & Fix

These three columns all measure **how much of available credit is being used**
as a percentage. Valid range is 0–100%. Values above 100% occur when a 
borrower exceeds their credit limit due to fees, interest or overspending.

---

### Column Explanations

**`revolving_utilisation`**
Percentage of revolving credit (credit cards, overdrafts) currently in use.

**`overall_utilisation`**
Percentage of ALL credit (both revolving and instalment) currently in use.

**`credit_card_utilisation_pct`**
Percentage of bank card (credit card) specific credit currently in use.

In [0]:
# Check distribution of utilisation columns
for c in ['revolving_utilisation', 'overall_utilisation', 'credit_card_utilisation_pct']:
    print(f"{c}:")
    df_clean.select(
        min(c).alias('min'),
        percentile_approx(c, 0.50).alias('median'),
        percentile_approx(c, 0.99).alias('p99'),
        max(c).alias('max')
    ).show()
    print(f"  Values above 100: {df_clean.filter(col(c) > 100).count():,}")
    print()

---

### Distribution Findings

| Column | Median | p99 | Max | Values > 100% |
|--------|--------|-----|-----|---------------|
| `revolving_utilisation` | 48.2% | 98.2% | 193.0% | 6,039 |
| `overall_utilisation` | 50.0% | 100.0% | 239.0% | 16,795 |
| `credit_card_utilisation_pct` | 57.5% | 100.5% | 318.2% | 22,789 |

---

### Key Observations

- All three columns have **healthy medians between 48–58%** — typical 
  for active borrowers
- Values above 100% are **real** — borrowers exceeding their credit limit
- However they represent a small minority of records
- Maximum values (193%, 239%, 318%) would severely distort Power BI axes

---

### Decision — Cap all three at 100%

**Why not set to NULL?**
Values above 100% are real financial data — the borrower genuinely 
exceeded their limit. Setting to NULL would lose this signal.

**Why cap at 100%?**
- Preserves the meaning — "at or over credit limit"
- Prevents Power BI axis stretching to 318%
- Standard practice for utilisation metrics in credit analytics
- Consistent treatment across all three utilisation columns

**Action taken:** All values above 100% capped at 100%

In [0]:
# ---------------------------------------------------------------
# Cap utilisation columns at 100%
# ---------------------------------------------------------------
print("Step 2 — Capping utilisation columns at 100%:")
print("-" * 60)

util_cols = [
    'revolving_utilisation',
    'overall_utilisation',
    'credit_card_utilisation_pct'
]

for c in util_cols:
    if c in df_clean.columns:
        high = df_clean.filter(col(c) > 100).count()
        df_clean = df_clean.withColumn(
            c, when(col(c) > 100, 100).otherwise(col(c))
        )
        print(f"  ✓ {c:<35} {high:,} values capped at 100")

print()

# Verify
print("Verification after capping:")
for c in util_cols:
    if c in df_clean.columns:
        df_clean.select(
            min(c).alias('min'),
            percentile_approx(c, 0.99).alias('p99'),
            max(c).alias('max')
        ).show()
        print(f"  {c} — max should be 100")

### Count Columns — Distribution Analysis & Fix

These three columns count **specific negative credit events** in a 
borrower's credit history. They are whole number counts — 0 means 
the event never occurred.

---

### Column Explanations

**`open_credit_accounts`**
Total number of currently open credit accounts the borrower has.
Includes credit cards, loans, overdrafts — any active credit line.

**`derogatory_public_records`**
Number of negative public records on the borrower's credit file.
Examples: bankruptcies, court judgements, tax liens, civil suits.
Most borrowers have 0 — any value above 0 is a red flag.

**`tax_lien_records`**
Number of tax liens recorded against the borrower.
A tax lien means the government has a legal claim on the borrower's 
property due to unpaid taxes. Serious negative credit event.



In [0]:
# Check distribution of count columns
for c in ['open_credit_accounts', 'derogatory_public_records', 'tax_lien_records']:
    print(f"{c}:")
    df_clean.select(
        min(c).alias('min'),
        percentile_approx(c, 0.50).alias('median'),
        percentile_approx(c, 0.95).alias('p95'),
        percentile_approx(c, 0.99).alias('p99'),
        max(c).alias('max')
    ).show()
    print(f"  Values above p99: {df_clean.filter(col(c) > df_clean.select(percentile_approx(c, 0.99)).collect()[0][0]).count():,}")
    print()

# Check policy_code
print("policy_code distinct values:")
df_clean.select('policy_code').distinct().show()


### Distribution Findings

| Column | Median | p95 | p99 | Max | Values above p99 |
|--------|--------|-----|-----|-----|-----------------|
| `open_credit_accounts` | 11 | 23 | 30 | 101 | 17,111 |
| `derogatory_public_records` | 0 | 1 | 2 | 86 | 15,631 |
| `tax_lien_records` | 0 | 0 | 2 | 85 | 7,573 |

---

### Key Observations

**`open_credit_accounts`:**
- Median of 11 — typical borrower has 11 open accounts
- p99 = 30 — 99% of borrowers have 30 or fewer accounts
- Max of 101 — extreme outlier, likely data entry error
- Values above 30 distort scatter plot axes in Power BI

**`derogatory_public_records`:**
- Median of 0 — most borrowers have no public records
- p99 = 2 — 99% of borrowers have 2 or fewer
- Max of 86 — impossible, likely sentinel value
- Even 10+ public records is extremely unusual

**`tax_lien_records`:**
- Median of 0 — most borrowers have no tax liens
- p99 = 2 — 99% of borrowers have 2 or fewer
- Max of 85 — impossible, likely sentinel value
- Any value above 5 is highly suspicious

---

### `policy_code` — Zero Variation

| Value | Count |
|-------|-------|
| 1.0 | All 1,792,240 rows |

Every single row has `policy_code = 1.0` — zero variation. 
This column adds no analytical value whatsoever and will be dropped.

---

### Decisions

| Column | Action | Reason |
|--------|--------|--------|
| `open_credit_accounts` | Cap at p99 (30) | Values above 30 distort visuals |
| `derogatory_public_records` | Cap at p99 (2) | Max of 86 is impossible |
| `tax_lien_records` | Cap at p99 (2) | Max of 85 is impossible |
| `policy_code` | Drop | Zero variation — useless for analysis |

In [0]:
# ---------------------------------------------------------------
# Step 3 — Cap count columns at 99th percentile
# ---------------------------------------------------------------
print("Step 3 — Capping count columns at 99th percentile:")
print("-" * 60)

cap_cols = [
    'open_credit_accounts',
    'derogatory_public_records',
    'tax_lien_records'
]

for c in cap_cols:
    if c in df_clean.columns:
        # Use percentile_approx — avoids lazy evaluation issue
        p99 = df_clean.select(
            percentile_approx(c, 0.99).alias('p99')
        ).collect()[0]['p99']

        high = df_clean.filter(col(c) > p99).count()

        df_clean = df_clean.withColumn(
            c, when(col(c) > p99, p99).otherwise(col(c))
        )
        print(f"  ✓ {c:<35} {high:,} values capped at p99: {p99}")

print()

# Verify
print("Verification after capping:")
for c in cap_cols:
    if c in df_clean.columns:
        df_clean.select(
            min(c).alias('min'),
            percentile_approx(c, 0.99).alias('p99'),
            max(c).alias('max')
        ).show()

# ---------------------------------------------------------------
# Step 4 — Drop policy_code
# ---------------------------------------------------------------
print("Step 4 — Dropping zero-variation columns:")
print("-" * 60)

if 'policy_code' in df_clean.columns:
    df_clean = df_clean.drop('policy_code')
    print("  ✓ policy_code dropped — all values = 1.0, zero analytical value")

print()
print(f"✓ Outlier fixing complete")
print(f"  Rows    : {df_clean.count():,}")
print(f"  Columns : {len(df_clean.columns)}")

In [0]:
# ======================================================================
# EDA: Default Rate by Grade
# ======================================================================

print("EDA: Default Rate by Grade")
print("=" * 80)

# Calculate default rate per grade
default_by_grade = df_clean.groupBy('grade').agg(
    count('*').alias('total_loans'),
    sum(when(col('loan_status').isin(
        ['Charged Off', 'Default']), 1).otherwise(0)
    ).alias('defaults')
).withColumn(
    'default_rate',
    round(col('defaults') / col('total_loans') * 100, 2)
).orderBy('grade')

print("Default rate by grade:")
default_by_grade.show()

# Plot
grade_pd = default_by_grade.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Default Rate by Grade', fontsize=14, fontweight='bold')

# Bar chart — default rate
colors = ['green' if r < 5
          else 'orange' if r < 15
          else 'red'
          for r in grade_pd['default_rate']]

bars = axes[0].bar(grade_pd['grade'], grade_pd['default_rate'],
                   color=colors, edgecolor='white')
axes[0].set_title('Default Rate % by Grade')
axes[0].set_xlabel('Grade')
axes[0].set_ylabel('Default Rate (%)')

for bar, rate in zip(bars, grade_pd['default_rate']):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{rate}%', ha='center', fontsize=10)

# Bar chart — loan volume
axes[1].bar(grade_pd['grade'], grade_pd['total_loans'],
            color='steelblue', edgecolor='white')
axes[1].set_title('Loan Volume by Grade')
axes[1].set_xlabel('Grade')
axes[1].set_ylabel('Number of Loans')
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
)

plt.tight_layout()
plt.show()


### Key Findings

| Grade | Default Rate | Risk Level | Loan Volume |
|-------|-------------|------------|-------------|
| A | 2.76% | 🟢 Low | 360,000+ |
| B | 7.05% | 🟢 Low | 520,000+ |
| C | 11.97% | 🟠 Medium | 520,000+ |
| D | 17.18% | 🟠 Medium | 245,000+ |
| E | 25.07% | 🔴 High | 100,000+ |
| F | 34.47% | 🔴 High | 28,000+ |
| G | 36.95% | 🔴 High | 8,000+ |

---

### Observations

**Clear risk gradient confirmed:**
- Grade A is the safest — only 2.76% default rate
- Default rate increases consistently from A → G
- Grade G is 13x more likely to default than Grade A
- Grades F and G exceed 34% default rate — extremely high risk

**Loan volume vs risk:**
- Majority of loans are Grade B and C — moderate risk
- Very few Grade F and G loans issued — lender correctly
  limits exposure to highest risk borrowers
- Grade A has lower volume than B/C — safer borrowers
  are harder to find

---

### Validation for Feature Engineering

This chart **confirms the `risk_band` feature boundaries are correct:**

| risk_band | Grades | Default Rate Range |
|-----------|--------|-------------------|
| Low | A, B | 2.76% — 7.05% |
| Medium | C, D | 11.97% — 17.18% |
| High | E, F, G | 25.07% — 36.95% |

The boundaries create three clearly distinct risk tiers with 
no overlap — validating the A-B / C-D / E-G grouping decision.

In [0]:
# ======================================================================
# EDA: Loan Status Distribution
# ======================================================================

print("EDA: Loan Status Distribution")
print("=" * 80)

# Data
status_pd = df_clean.groupBy('loan_status') \
                    .count() \
                    .orderBy('count', ascending=False) \
                    .toPandas()

status_pd['count'] = status_pd['count'].astype(int)
status_pd['pct'] = (status_pd['count'] / status_pd['count'].sum() * 100).round(2)

print(status_pd.to_string(index=False))
print()

# Totals
total = int(status_pd['count'].sum())
charged_off = int(status_pd.loc[status_pd['loan_status'] == 'Charged Off', 'count'].sum())
default_count = int(status_pd.loc[status_pd['loan_status'] == 'Default', 'count'].sum())
defaults = charged_off + default_count
performing = total - defaults

print(f"Total    : {total:,}")
print(f"Defaults : {defaults:,}")
print(f"Performing: {performing:,}")
print()

# Lists
status_list = list(status_pd['loan_status'])
count_list  = list(status_pd['count'])
pct_list    = list(status_pd['pct'])

# Colors
colors = []
for s in status_list:
    if s == 'Fully Paid':
        colors.append('green')
    elif s in ['Charged Off', 'Default']:
        colors.append('red')
    elif 'Late' in s:
        colors.append('orange')
    else:
        colors.append('steelblue')

# Plot 1 — bar chart
fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.barh(status_list, count_list, color=colors)
ax1.set_title('Loan Status Distribution')
ax1.set_xlabel('Number of Loans')
plt.tight_layout()
plt.show()

# Plot 2 — pie chart
fig2, ax2 = plt.subplots(figsize=(6, 6))
ax2.pie(
    [defaults, performing],
    labels=['Default', 'Performing'],
    autopct='%1.1f%%',
    colors=['red', 'green']
)
ax2.set_title('Default vs Performing')
plt.tight_layout()
plt.show()


### Key Findings

| Loan Status | Count | % | Category |
|-------------|-------|---|----------|
| Current | 866,379 | 48.34% | 🔵 Active |
| Fully Paid | 702,175 | 39.18% | 🟢 Performing |
| Charged Off | 190,108 | 10.61% | 🔴 Default |
| Late (31-120 days) | 21,028 | 1.17% | 🟠 At Risk |
| In Grace Period | 8,235 | 0.46% | 🟠 At Risk |
| Late (16-30 days) | 4,276 | 0.24% | 🟠 At Risk |
| Default | 39 | 0.00% | 🔴 Default |

---

### Observations

**Portfolio is largely healthy:**
- **89.4% performing** — Current + Fully Paid loans dominate
- **10.6% defaulted** — Charged Off is the primary default category
- `Default` status has only 39 loans — extremely rare

**At-risk loans:**
- Late and In Grace Period loans total **~1.87%** of portfolio
- These are potential future defaults — worth monitoring
- Combined they represent **33,539 loans** at risk

---

### Validation for Feature Engineering

**`default_flag` logic confirmed correct:**

In [0]:
# ======================================================================
# EDA: Annual Income Distribution
# ======================================================================

print("EDA: Annual Income Distribution")
print("=" * 80)

# Stats
df_clean.select(
    min('annual_income').alias('min'),
    percentile_approx('annual_income', 0.25).alias('Q1'),
    percentile_approx('annual_income', 0.50).alias('median'),
    percentile_approx('annual_income', 0.75).alias('Q3'),
    percentile_approx('annual_income', 0.95).alias('p95'),
    percentile_approx('annual_income', 0.99).alias('p99'),
    max('annual_income').alias('max')
).show()

# Quartile boundaries — used for income_band feature
quantiles = df_clean.approxQuantile('annual_income', [0.25, 0.50, 0.75], 0.01)
q1, q2, q3 = quantiles[0], quantiles[1], quantiles[2]

print(f"Quartile boundaries for income_band feature:")
print(f"  Q1 boundary : £{q1:,.0f}")
print(f"  Q2 boundary : £{q2:,.0f}")
print(f"  Q3 boundary : £{q3:,.0f}")
print()

# Count per income band
print("Income band distribution:")
df_clean.withColumn(
    'income_band_preview',
    when(col('annual_income') <= q1, 'Q1 - Low')
    .when(col('annual_income') <= q2, 'Q2 - Lower Middle')
    .when(col('annual_income') <= q3, 'Q3 - Upper Middle')
    .otherwise('Q4 - High')
).groupBy('income_band_preview') \
 .count() \
 .orderBy('income_band_preview') \
 .show()

# Plot
sample_pd = df_clean.select('annual_income') \
                    .filter(col('annual_income') <= 300000) \
                    .dropna() \
                    .sample(fraction=0.05, seed=42) \
                    .toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Annual Income Distribution',
             fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(sample_pd['annual_income'], bins=100,
             color='purple', edgecolor='white', alpha=0.7)
axes[0].set_title('Distribution (£0–£300K)')
axes[0].set_xlabel('Annual Income (£)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}')
)

# Add quartile lines
for q, label, color in zip(
        [q1, q2, q3],
        ['Q1', 'Q2\n(Median)', 'Q3'],
        ['blue', 'green', 'orange']):
    if q <= 300000:
        axes[0].axvline(x=q, color=color,
                        linestyle='--', linewidth=1.5)
        axes[0].text(q + 1000, axes[0].get_ylim()[1] * 0.9,
                     f'{label}\n£{q:,.0f}',
                     fontsize=8, color=color)

# Box plot
axes[1].boxplot(sample_pd['annual_income'],
                vert=False, patch_artist=True,
                boxprops=dict(facecolor='purple', alpha=0.7))
axes[1].set_title('Box Plot (£0–£300K)')
axes[1].set_xlabel('Annual Income (£)')
axes[1].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}')
)

plt.tight_layout()
plt.show()

# Outlier check
print("Income outlier check:")
print(f"  Income > £200K : {df_clean.filter(col('annual_income') > 200000).count():,} rows")
print(f"  Income > £500K : {df_clean.filter(col('annual_income') > 500000).count():,} rows")
print(f"  Income > £1M   : {df_clean.filter(col('annual_income') > 1000000).count():,} rows")



### Key Statistics

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Min | £0 | Some borrowers reported zero income |
| Q1 | £46,700 | 25% of borrowers earn below this |
| Median | £65,000 | Typical borrower income |
| Q3 | £95,000 | 75% of borrowers earn below this |
| p95 | £165,000 | Top 5% threshold |
| p99 | £280,000 | Top 1% threshold |
| Max | £110,000,000 | Extreme outlier |

---

### Income Band Distribution

| Band | Income Range | Count | % |
|------|-------------|-------|---|
| Q1 - Low | £0 — £46,700 | 445,376 | 24.8% |
| Q2 - Lower Middle | £46,700 — £65,000 | 460,469 | 25.7% |
| Q3 - Upper Middle | £65,000 — £95,000 | 450,390 | 25.1% |
| Q4 - High | £95,000+ | 436,005 | 24.3% |

---

### Observations

**Distribution shape:**
- Strong right skew — most borrowers earn £30,000–£100,000
- Peak around £50,000–£70,000 — typical UK salary range
- Long tail extending to £300,000+ — high earners present

**Quartile boundaries are well balanced:**
- All four income bands contain roughly 25% of borrowers
- Confirms quartile approach produces meaningful equal-sized groups
- No single band dominates the dataset

**Outliers:**
- 45,534 borrowers earn above £200,000 (2.5%)
- 2,994 borrowers earn above £500,000 (0.17%)
- 523 borrowers earn above £1,000,000 (0.03%)
- Max of £110M kept as-is — bucketed into Q4 High

**Box plot confirms:**
- Tight interquartile range (£46,700–£95,000)
- Many dots beyond whisker = large num

## FEATURE ENGINEERING

In [0]:
# ====================================================================
# Feature 1 - risk_band
# ====================================================================

print(" Creating risk_band ")
print("="* 80)
print("Logic: A-B = Low, C-D = Medium, E-F = High")
print()

df_features = df_clean.withColumn('risk_band',
                   when(col('grade').isin(['A','B']),'Low')
                   .when(col('grade').isin(['C','D']),'Medium')
                   .when(col('grade').isin(['E','F']),'High')
                   .when(col('grade')=='G','Very High')
                   .otherwise(None)
                   )

print("risk_band column created")
print()
print("Distribution")
df_features.groupBy('grade','risk_band')\
    .count()\
        .orderBy('grade')\
            .show()

In [0]:
# ======================================================================
# Feature 2: dti_category
# ======================================================================

print("Feature 2: dti_category")
print("-" * 60)
print("Logic: Low <15%, Medium 15-30%, High >30%")
print()

df_features = df_features.withColumn(
    'dti_category',
    when(col('debt_to_income') < 15, 'Low')
    .when(col('debt_to_income').between(15, 30), 'Medium')
    .when(col('debt_to_income') > 30, 'High')
    .otherwise(None)
)

print("✓ dti_category created")
print()
print("Distribution:")
df_features.groupBy('dti_category') \
           .count() \
           .orderBy('dti_category') \
           .show()


In [0]:
# ======================================================================
# Feature 3: loan_performance
# ======================================================================

print("Feature 3: loan_performance")
print("-" * 60)
print("Logic: Paid / Current / Late / Charged Off / Default")
print()

df_features = df_features.withColumn(
    'loan_performance',
    when(col('loan_status') == 'Fully Paid', 'Paid')
    .when(col('loan_status') == 'Current', 'Current')
    .when(col('loan_status').isin([
        'Late (16-30 days)',
        'Late (31-120 days)',
        'In Grace Period'
    ]), 'Late')
    .when(col('loan_status') == 'Charged Off', 'Charged Off')
    .when(col('loan_status') == 'Default', 'Default')
    .otherwise('Other')
)

print("✓ loan_performance created")
print()
print("Distribution:")
df_features.groupBy('loan_status', 'loan_performance') \
           .count() \
           .orderBy('loan_performance') \
           .show(truncate=False)

In [0]:
# ======================================================================
# Feature 4: default_flag
# ======================================================================

print("Feature 4: default_flag")
print("-" * 60)
print("Logic: 1 = Charged Off or Default, 0 = everything else")
print()

df_features = df_features.withColumn(
    'default_flag',
    when(
        col('loan_status').isin(['Charged Off', 'Default']), 1
    ).otherwise(0)
)

print("✓ default_flag created")
print()
print("Distribution:")
df_features.groupBy('default_flag') \
           .count() \
           .orderBy('default_flag') \
           .show()

# Default rate
total = df_features.count()
defaults = df_features.filter(col('default_flag') == 1).count()
default_rate = builtins.round(defaults / total * 100, 2)
print(f"Overall default rate: {default_rate}%")

In [0]:
# ======================================================================
# Feature 5: income_band
# ======================================================================

print("Feature 5: income_band")
print("-" * 60)
print("Logic: Quartile grouping based on annual_income")
print()

# Use confirmed quartile boundaries from EDA
q1 = 46700
q2 = 65000
q3 = 95000

print(f"  Q1 boundary : £{q1:,}")
print(f"  Q2 boundary : £{q2:,}")
print(f"  Q3 boundary : £{q3:,}")
print()

df_features = df_features.withColumn(
    'income_band',
    when(col('annual_income') <= q1, 'Q1 - Low')
    .when(col('annual_income') <= q2, 'Q2 - Lower Middle')
    .when(col('annual_income') <= q3, 'Q3 - Upper Middle')
    .otherwise('Q4 - High')
)

print("✓ income_band created")
print()
print("Distribution:")
df_features.groupBy('income_band') \
           .count() \
           .orderBy('income_band') \
           .show()

In [0]:
# ======================================================================
# Feature 6: loan_age_months
# ======================================================================

print("Feature 6: loan_age_months")
print("-" * 60)
print("Logic: Months between issue_date and last_pymnt_d")
print()

df_features = df_features.withColumn(
    'loan_age_months',
    when(
        col('last_pymnt_d').isNotNull(),
        months_between(
            col('last_pymnt_d'), col('issue_date')
        ).cast(IntegerType())
    ).otherwise(None)
)

print("✓ loan_age_months created")
print()
print("Summary stats:")
df_features.select(
    mean('loan_age_months').alias('avg_months'),
    min('loan_age_months').alias('min_months'),
    max('loan_age_months').alias('max_months')
).show()

In [0]:
# ======================================================================
# Feature 7: recovery_rate
# ======================================================================

print("Feature 7: recovery_rate")
print("-" * 60)
print("Logic: recoveries / funded_amount x 100 — defaulted loans only")
print()

df_features = df_features.withColumn(
    'recovery_rate',
    when(
        (col('default_flag') == 1) & (col('funded_amount') > 0),
        round(col('recoveries') / col('funded_amount') * 100, 2)
    ).otherwise(None)
)

print("✓ recovery_rate created")
print()
print("Recovery rate summary (defaulted loans only):")
df_features.filter(col('default_flag') == 1) \
           .select(
               mean('recovery_rate').alias('avg_recovery_rate'),
               min('recovery_rate').alias('min_recovery_rate'),
               max('recovery_rate').alias('max_recovery_rate')
           ).show()

In [0]:
# ======================================================================
# Feature 8: approval_flag
# ======================================================================

print("Feature 8: approval_flag")
print("-" * 60)
print("Logic: 1 = Accepted loan, 0 = Rejected application")
print()

df_features = df_features.withColumn('approval_flag', lit(1))

print("✓ approval_flag created")
print("  All records = accepted loans → approval_flag = 1")
print("  Rejected loans (0) will be added in Notebook 04")
print()
df_features.groupBy('approval_flag').count().show()

In [0]:
# ======================================================================
# Feature Engineering Summary
# ======================================================================

print("FEATURE ENGINEERING SUMMARY")
print("=" * 80)

new_features = [
    'risk_band', 'dti_category', 'loan_performance',
    'default_flag', 'income_band', 'loan_age_months',
    'recovery_rate', 'approval_flag'
]

print(f"New features created: {len(new_features)}")
print()

for f in new_features:
    dtype = dict(df_features.dtypes)[f]
    distinct = df_features.select(f).distinct().count()
    nulls = df_features.filter(col(f).isNull()).count()
    print(f"  {f:<25} type: {dtype:<10} distinct: {distinct:<5} nulls: {nulls:,}")

print()
print(f"Total columns after feature engineering: {len(df_features.columns)}")

## Feature Engineering — Completed ✓

8 new business features created from existing columns to enable 
advanced analytics in Snowflake and Power BI.

---

### 1. `risk_band` — Risk Classification
**Source:** `grade` | **Type:** String | **Distinct:** 4 | **Nulls:** 0

| Grade | Risk Band | Default Rate |
|-------|-----------|-------------|
| A, B | Low | 2.76% – 7.05% |
| C, D | Medium | 11.97% – 17.18% |
| E, F | High | 25.07% – 34.47% |
| G | Very High | 36.95% |

4 tiers chosen over 3 based on EDA — Grade G has a meaningfully 
different default rate from E and F, justifying its own tier.

---

### 2. `dti_category` — Affordability Assessment
**Source:** `debt_to_income` | **Type:** String | **Distinct:** 3 | **Nulls:** 0

| DTI Range | Category |
|-----------|----------|
| < 15% | Low |
| 15% – 30% | Medium |
| > 30% | High |

Industry standard thresholds used by UK banks to assess borrower 
affordability. High DTI = less disposable income = higher default risk.

---

### 3. `loan_performance` — Portfolio Health Flag
**Source:** `loan_status` | **Type:** String | **Distinct:** 5 | **Nulls:** 0

| Loan Status | Performance |
|-------------|-------------|
| Fully Paid | Paid |
| Current | Current |
| Late (16-30d), Late (31-120d), In Grace Period | Late |
| Charged Off | Charged Off |
| Default | Default |

Groups 7 granular loan statuses into 5 meaningful performance 
categories for executive reporting.

---

### 4. `default_flag` — Risk Binary Flag
**Source:** `loan_status` | **Type:** Integer | **Distinct:** 2 | **Nulls:** 0

| Value | Meaning | Count |
|-------|---------|-------|
| 1 | Charged Off or Default | 190,147 (10.6%) |
| 0 | Performing | 1,602,093 (89.4%) |

Core metric for all risk analysis — enables default rate 
calculations across any dimension (grade, state, purpose, year).

---

### 5. `income_band` — Borrower Wealth Profile
**Source:** `annual_income` | **Type:** String | **Distinct:** 4 | **Nulls:** 0

| Income Range | Band | Count |
|-------------|------|-------|
| ≤ £46,700 | Q1 - Low | 445,376 |
| £46,700 – £65,000 | Q2 - Lower Middle | 460,469 |
| £65,000 – £95,000 | Q3 - Upper Middle | 450,390 |
| > £95,000 | Q4 - High | 436,005 |

Quartile boundaries confirmed by EDA. Neutralises £110M extreme 
outlier by bucketing into Q4 — no capping needed.

---

### 6. `loan_age_months` — Lifecycle Tracking
**Source:** `issue_date`, `last_pymnt_d` | **Type:** Integer | **Distinct:** 51 | **Nulls:** 

months_between(last_pymnt_d, issue_date)
Range: 0 – 50 months | Average: 17.6 months

Tracks how long a loan has been active. Essential for cohort 
analysis — used to identify when defaults typically peak 
relative to loan issue date.

---

### 7. `recovery_rate` — Loss Recovery KPI
**Source:** `recoveries`, `funded_amount` | **Type:** Double | **Nulls:** 1,602,093
recovery_rate = (recoveries / funded_amount) × 100
Calculated for defaulted loans only (default_flag = 1)
NULL for performing loans — not applicable

Measures how much of a defaulted loan was recovered through 
collections. NULLs are expected and meaningful — recovery only 
applies when a loan has charged off.

---

### 8. `approval_flag` — Funnel Analysis Indicator
**Source:** Hardcoded | **Type:** Integer | **Distinct:** 1 | **Nulls:** 0

Accepted loans (this dataset) → 1
Rejected applications         → 0 (added in Notebook 04)

Enables full funnel analysis when combined with rejected loans 
in Notebook 04. All records here are accepted → flag = 1.

---

### Summary

| Feature | Type | Distinct | Nulls | Source |
|---------|------|----------|-------|--------|
| `risk_band` | String | 4 | 0 | `grade` |
| `dti_category` | String | 3 | 0 | `debt_to_income` |
| `loan_performance` | String | 5 | 0 | `loan_status` |
| `default_flag` | Integer | 2 | 0 | `loan_status` |
| `income_band` | String | 4 | 0 | `annual_income` |
| `loan_age_months` | Integer | 51 | 0 | `issue_date`, `last_pymnt_d` |
| `recovery_rate` | Double | 5,715 | 1,602,093 | `recoveries`, `funded_amount` |
| `approval_flag` | Integer | 1 | 0 | Hardcoded |

**Total columns after feature engineering: 109**
**Enriched dataset saved to Delta Lake — ready for Notebook 04**


In [0]:
# ======================================================================
# Save Enriched Dataset as Delta Table
# ======================================================================

print("Saving Enriched Dataset to Delta Lake")
print("=" * 80)

enriched_path = "/Volumes/workspace/lending_club/lending_club_data/enriched_loans_delta"

# Save
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .save(enriched_path)

print(f"✓ Enriched Delta table saved")
print(f"  Path    : {enriched_path}")
print(f"  Rows    : {df_features.count():,}")
print(f"  Columns : {len(df_features.columns)}")
print()

# Validate — reload and confirm
df_check = spark.read.format("delta").load(enriched_path)

print("Validation:")
print(f"  Reloaded rows    : {df_check.count():,}")
print(f"  Reloaded columns : {len(df_check.columns)}")
print()

# Confirm new features exist
print("Confirming new features in saved table:")
new_features = ['risk_band', 'dti_category', 'loan_performance',
                'default_flag', 'income_band', 'loan_age_months',
                'recovery_rate', 'approval_flag']

for f in new_features:
    exists = f in df_check.columns
    print(f"  {f:<25} {'✓' if exists else '✗'}")